# 🏁 UIS Grand Prix Foundationals Models — Servidor GPU (Colab A100 / L4)

Ejecuta estas dos celdas en orden:
1. **Entorno:** `Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> GPU A100`.
2. **Celda 1:** Clona el repositorio, instala dependencias y descarga `cloudflared`.
3. **Celda 2:** Inicia `uvicorn` (FastAPI + `Qwen/Qwen2.5-1.5B-Instruct`), abre el túnel público `trycloudflare.com` y mantiene la celda viva mostrando el estado cada 60 s.

In [1]:
# Celda 1: Verificar GPU, clonar/actualizar repositorio e instalar dependencias
!nvidia-smi

import os
REPO_DIR = "/content/Foundational-models"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/DaniDotExe/Foundational-models.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only

!pip -q uninstall -y torchao
!pip -q install -r {REPO_DIR}/backend_server/requirements.txt
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
print("Listo. Continúa con la Celda 2.")

Wed Nov 18 17:01:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   63C    P8             18W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Celda 2: Iniciar FastAPI (--workers 1) + túnel trycloudflare.com + keep-alive
import json
import os
import re
import shutil
import subprocess
import time
import urllib.request

BACKEND_DIR = "/content/Foundational-models/backend_server"
os.chdir(BACKEND_DIR)
if not os.path.exists(".env") and os.path.exists(".env.example"):
    shutil.copy(".env.example", ".env")

# Detener instancias previas si se re-ejecuta la celda
subprocess.run(["pkill", "-9", "-f", "uvicorn server:app"], check=False)
subprocess.run(["pkill", "-9", "-f", "cloudflared tunnel"], check=False)
time.sleep(2)

server_log = open("/content/uvicorn.log", "w")
server_proc = subprocess.Popen(
    ["python3", "-m", "uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "1"],
    stdout=server_log,
    stderr=subprocess.STDOUT,
)

print("Cargando Qwen/Qwen2.5-1.5B-Instruct en GPU...")
salud_ok = False
for _ in range(120):
    if server_proc.poll() is not None:
        print("ERROR: uvicorn terminó inesperadamente. Últimas líneas de /content/uvicorn.log:")
        print(open("/content/uvicorn.log", errors="ignore").read()[-2000:])
        raise RuntimeError("Fallo al iniciar uvicorn")
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2) as r:
            if r.status == 200:
                print("Backend listo:", r.read().decode())
                salud_ok = True
                break
    except Exception:
        time.sleep(2)

if not salud_ok:
    print(open("/content/uvicorn.log", errors="ignore").read()[-2000:])
    raise TimeoutError("El servidor no respondió en /health a tiempo.")

cf_log = open("/content/cloudflared.log", "w")
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=cf_log,
    stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(45):
    time.sleep(1)
    txt = open("/content/cloudflared.log", errors="ignore").read()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", txt)
    if m:
        public_url = m.group(0)
        break

if not public_url:
    print(open("/content/cloudflared.log", errors="ignore").read()[-2000:])
    raise RuntimeError("No se pudo obtener la URL de trycloudflare.com")

print("\n" + "=" * 72)
print(f"  SERVER_URL={public_url}")
print(f"  RANKING EN VIVO: {public_url}/ranking")
print(f"  HEALTH CHECK:    {public_url}/health")
print("=" * 72 + "\n")
print("Mantén esta celda corriendo (Ctrl+C o Stop para apagar).")

# Loop keep-alive para monitorear en vivo quién entrena y quién simula
last_pos = 0
last_health = time.monotonic()
try:
    while True:
        time.sleep(2)
        if os.path.exists("/content/uvicorn.log"):
            with open("/content/uvicorn.log", "r", errors="ignore") as lf:
                lf.seek(last_pos)
                nuevas = lf.read()
                last_pos = lf.tell()
            for ln in nuevas.splitlines():
                if any(tag in ln for tag in ("[ENTRENANDO]", "[LORA LISTO]", "[LORA ERROR]", "[SIMULANDO]", "LoRA entrenado")):
                    print(ln, flush=True)
        if server_proc.poll() is not None:
            print("[ALERTA] uvicorn se detuvo. Revisa /content/uvicorn.log")
            break
        if time.monotonic() - last_health >= 60:
            last_health = time.monotonic()
            try:
                with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=5) as r:
                    info = json.loads(r.read().decode())
                print(
                    f"[{time.strftime('%H:%M:%S')}] OK | {info.get('gpu_nombre')} | "
                    f"VRAM libre: {info.get('vram_libre_mb')} MB | "
                    f"Adaptadores: {info.get('adaptadores_en_memoria')}",
                    flush=True,
                )
            except Exception as exc:
                print(f"[{time.strftime('%H:%M:%S')}] Aviso /health: {exc}", flush=True)
except KeyboardInterrupt:
    print("Deteniendo servidor y túnel...")
    cf_proc.terminate()
    server_proc.terminate()

Cargando Qwen/Qwen2.5-1.5B-Instruct en GPU...
Backend listo: {"status":"ok","mock_mode":false,"model_id":"Qwen/Qwen2.5-1.5B-Instruct","device":"cuda","torch_dtype":"torch.bfloat16","gpu_activa":true,"gpu_nombre":"NVIDIA L4","vram_libre_mb":19235.1,"vram_total_mb":22563.1,"adaptadores_en_memoria":[],"adaptadores_en_disco":["1","2","3","4","5","6","dani"],"limites":{"max_k_shots":30,"max_lora_epochs":15,"max_lora_r":32,"latency_penalty_weight":1.5}}

  SERVER_URL=https://their-secretariat-converter-www.trycloudflare.com
  RANKING EN VIVO: https://their-secretariat-converter-www.trycloudflare.com/ranking
  HEALTH CHECK:    https://their-secretariat-converter-www.trycloudflare.com/health

Mantén esta celda corriendo (Ctrl+C o Stop para apagar).
[2026-11-18 17:02:10,458] INFO grandprix_server: [ENTRENANDO] dani | ejemplos=283 | epochs=8 | r=16
[17:02:48] OK | NVIDIA L4 | VRAM libre: 12521.1 MB | Adaptadores: []
[17:03:48] OK | NVIDIA L4 | VRAM libre: 12521.1 MB | Adaptadores: []
[2026-11-18